# PlateDetectionSystem — YOLOv11s Egitim Notebook
## Otopark Plaka Tespit Modeli — Global Dataset

**Gereksinimler:**
1. Google Colab'da **L4 GPU** aktif olsun: `Calisma Zamani > Calisma Zamani Turunu Degistir > L4 GPU`
2. Google Drive'iniza `yolo_plate_dataset_v2.zip` yuklensin — konum: `MyDrive/plaka/yolo_plate_dataset_v2.zip`
3. Bu notebook'un tum hucrelerini sirayla calistirin

**Beklenen Sure:** L4 GPU ile ~45-70 dakika (erken durdurma devreye girerse daha az)

**Otomatik Yedekleme:** Her 5 epoch'ta `MyDrive/PlateDetection_Models/checkpoints/` klasorune kaydedilir. Baglanti kesilse bile model kaybolmaz.

## Adim 0 — GPU Kontrolu

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA mevcut: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu}')
    print(f'VRAM: {vram:.1f} GB')
    print('Harika! Devam edebilirsiniz.')
else:
    print('HATA: GPU bulunamadi!')
    print('Cozum: Calisma Zamani > Calisma Zamani Turunu Degistir > T4 GPU')

## Adim 1 — Gerekli Kutuphaneleri Yukle

In [ ]:
!pip install ultralytics>=8.2.0 -q
print('Kurulum tamamlandi.')

## Adim 2 — Google Drive'i Bagla

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive baglandi.')

## Adim 3 — Dataset'i Ac

> **ZIP konumu:** `MyDrive/plaka/yolo_plate_dataset_v2.zip`
> Farkli bir klasore yuklediyseniz asagidaki `ZIP_PATH` degerini guncelleyin.

In [ ]:
import os
import zipfile

ZIP_PATH = '/content/drive/MyDrive/plaka/yolo_plate_dataset_v2.zip'
EXTRACT_DIR = '/content/'

if not os.path.exists(ZIP_PATH):
    print(f'HATA: Zip dosyasi bulunamadi: {ZIP_PATH}')
    print('Lutfen yolo_plate_dataset_v2.zip dosyasini Google Drive\'e yukleyin.')
else:
    print(f'Zip aciliyor: {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Zip acma tamamlandi!')

    dataset_path = '/content/yolo_plate_dataset_v2'
    train_imgs = len(os.listdir(f'{dataset_path}/images/train'))
    val_imgs   = len(os.listdir(f'{dataset_path}/images/val'))
    test_imgs  = len(os.listdir(f'{dataset_path}/images/test'))
    print(f'Train: {train_imgs} | Val: {val_imgs} | Test: {test_imgs}')

## Adim 4 — dataset.yaml Guncelle

Colab'in dosya yoluna gore yaml'i guncelliyoruz.

In [ ]:
dataset_yaml = '''
path: /content/yolo_plate_dataset_v2
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: plate
'''.strip()

with open('/content/yolo_plate_dataset_v2/dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

print('dataset.yaml guncellendi:')
print(dataset_yaml)

## Adim 5 — Model Egitimi

**YOLOv11s** modeli ile egitim baslatiliyor.

- Model: `yolo11s.pt` (Small variant, 9.4M parametre)
- Goruntu boyutu: `640x640`
- Epoch: `100` maksimum — erken durdurma: 20 epoch boyunca iyilesme olmazsa otomatik durur
- Optimizer: `AdamW`
- Otomatik yedek: her 5 epoch'ta `best.pt` Drive'a kopyalanir

**Tahmini sure: L4 GPU ile ~45-70 dakika**

In [ ]:
import torch
import shutil
import os
torch.cuda.empty_cache()

from ultralytics import YOLO

# Her 5 epoch'ta Drive'a otomatik yedek
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/PlateDetection_Models/checkpoints'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
SAVE_EVERY = 5  # kac epoch'ta bir yedekle

def backup_to_drive(trainer):
    epoch = trainer.epoch + 1
    if epoch % SAVE_EVERY == 0:
        src = '/content/runs/plate_det_global_v1/weights/last.pt'
        if os.path.exists(src):
            dst = f'{DRIVE_BACKUP_DIR}/last_epoch{epoch:03d}.pt'
            shutil.copy2(src, dst)
            print(f'[YEDEK] Epoch {epoch} -> Drive kaydedildi: {dst}')
    # best.pt her zaman guncelle
    best_src = '/content/runs/plate_det_global_v1/weights/best.pt'
    if os.path.exists(best_src):
        shutil.copy2(best_src, f'{DRIVE_BACKUP_DIR}/best.pt')

model = YOLO('yolo11s.pt')
model.add_callback('on_train_epoch_end', backup_to_drive)

results = model.train(
    data='/content/yolo_plate_dataset_v2/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,

    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,

    patience=20,

    degrees=15,
    perspective=0.002,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,

    box=7.5,
    cls=0.5,
    dfl=1.5,

    project='/content/runs',
    name='plate_det_global_v1',
    save_period=5,
    plots=True,
    verbose=True,
)

print('\nEgitim tamamlandi!')
print(f"mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

# Son kayit
shutil.copy2('/content/runs/plate_det_global_v1/weights/best.pt',
             '/content/drive/MyDrive/PlateDetection_Models/plate_det_global_v1_best.pt')
print('best.pt Drive ana klasorune kaydedildi!')

## Adim 6 — Test Seti Degerlendirme

In [ ]:
from ultralytics import YOLO

best_model = YOLO('/content/runs/plate_det_global_v1/weights/best.pt')

metrics = best_model.val(
    data='/content/yolo_plate_dataset_v2/dataset.yaml',
    split='test',
    imgsz=640,
    device=0,
)

print('\n=== TEST SETI SONUCLARI ===')
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")

## Adim 7 — Modeli Drive'a Kaydet

Egitilen modeli Google Drive'a kopyala. Sonra bilgisayarina indirip projeye ekle.

In [ ]:
import shutil
import os

src = '/content/runs/plate_det_global_v1/weights/best.pt'
dst_dir = '/content/drive/MyDrive/PlateDetection_Models'

os.makedirs(dst_dir, exist_ok=True)
shutil.copy2(src, f'{dst_dir}/plate_det_global_v1_best.pt')

# Egitim sonuclarini da kaydet
shutil.copy2(
    '/content/runs/plate_det_global_v1/results.csv',
    f'{dst_dir}/plate_det_global_v1_results.csv'
)

print(f'Model kaydedildi: {dst_dir}/plate_det_global_v1_best.pt')
print('Google Drive > PlateDetection_Models klasorunden indirebilirsiniz.')

## Adim 8 — ONNX Export (Opsiyonel ama Onerilen)

ONNX formatina export etmek CPU inference'i ~30% hizlandirir.
Bu modeli de Drive'a kaydediyoruz.

In [ ]:
from ultralytics import YOLO
import shutil

best_model = YOLO('/content/runs/plate_det_global_v1/weights/best.pt')

# ONNX export
onnx_path = best_model.export(
    format='onnx',
    simplify=True,
    imgsz=640,
    device=0,
)

dst_dir = '/content/drive/MyDrive/PlateDetection_Models'
shutil.copy2(str(onnx_path), f'{dst_dir}/plate_det_global_v1_best.onnx')

print(f'ONNX modeli kaydedildi: {dst_dir}/plate_det_global_v1_best.onnx')

## Tamamlandi!

**Sonraki adimlar:**
1. `plate_det_global_v1_best.pt` dosyasini Drive'dan bilgisayarina indir
2. `PlateDetectionSystem/models/` klasorune kopyala
3. `.env` dosyasinda `MODEL_PATH=models/plate_det_global_v1_best.pt` olarak guncelle
4. Web uygulamasini calistir: `python run.py`